# 📹 Video 2: Klasifikasi dengan Algoritma C4.5 (Decision Tree)

**Mata Kuliah**: Data Mining (21TIF604)  
**Universitas Islam Nahdlatul Ulama Jepara**  
**Dosen**: Ir. Adi Sucipto, M.Kom

---

### Tujuan Video Ini:
1. Memahami konsep algoritma C4.5 (Decision Tree) untuk klasifikasi
2. Menyiapkan fitur dan label dari data bersih hasil preprocessing
3. Membangun model klasifikasi: prediksi pelanggan **loyal** vs **tidak loyal**
4. Mengevaluasi akurasi model menggunakan Confusion Matrix
5. Memvisualisasikan pohon keputusan yang dihasilkan

---

### Konteks Bisnis (Renstra):
Kita akan mengklasifikasikan pelanggan menjadi **loyal** (sering belanja) dan **tidak loyal** (jarang belanja).  
Hasil klasifikasi ini dapat membantu bisnis retail mengidentifikasi pelanggan potensial dan memberikan perlakuan khusus (promo, diskon, program membership).

In [ ]:
# ============================================================
# STEP 1: INSTALL LIBRARY & MOUNT GOOGLE DRIVE
# ============================================================
# Google Colab sudah punya: pandas, numpy, matplotlib, seaborn, scikit-learn
# Yang perlu diinstall tambahan: mlxtend (untuk video 4, tapi install sekalian)
# ============================================================

!pip install mlxtend -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd                           # Manipulasi data
import numpy as np                            # Operasi numerik
import matplotlib.pyplot as plt               # Visualisasi grafik
import seaborn as sns                         # Visualisasi lebih cantik
import os                                     # Operasi sistem file

# Library dari scikit-learn untuk machine learning
from sklearn.model_selection import train_test_split    # Membagi data menjadi data latih & data uji
from sklearn.tree import DecisionTreeClassifier         # Algoritma Decision Tree (C4.5)
from sklearn.metrics import (                          # Metrik evaluasi model
    accuracy_score,                                     # Akurasi: seberapa benar prediksi keseluruhan
    confusion_matrix,                                   # Matriks kebingungan: TP, FP, FN, TN
    classification_report,                              # Laporan lengkap: precision, recall, f1-score
)
from sklearn import tree                                # Untuk visualisasi pohon keputusan

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)

# Folder kerja di Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/data-mining'

print("✅ Google Drive ter-mount & semua library berhasil diimport!")
print(f"📁 Folder kerja: {DRIVE_FOLDER}")

## Step 2: Load Data Bersih dari Preprocessing

Data bersih sudah disimpan di Video 1 (notebook `01_preprocessing.ipynb`).  
Kita load file CSV tersebut untuk digunakan di sini.

In [ ]:
# ============================================================
# LOAD DATA BERSIH HASIL PREPROCESSING (Video 1)
# ============================================================
# File ini dihasilkan oleh notebook 01_preprocessing.ipynb
# Pastikan sudah menjalankan notebook preprocessing terlebih dahulu
# File tersimpan di Google Drive
# ============================================================

data_path = os.path.join(DRIVE_FOLDER, 'online_retail_clean.csv')
df = pd.read_csv(data_path)

# Konversi InvoiceDate kembali ke datetime (karena CSV tidak menyimpan tipe data)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"✅ Data bersih berhasil dimuat!")
print(f"   Jumlah baris: {len(df):,}")
print(f"   Jumlah kolom: {df.shape[1]}")
print(f"\n📋 Kolom tersedia: {list(df.columns)}")
df.head()

## Step 3: Feature Engineering — Menyiapkan Fitur dan Label

Untuk klasifikasi, kita perlu:
- **Fitur (X)**: variabel independen yang digunakan untuk memprediksi
- **Label (y)**: variabel dependen yang ingin diprediksi

**Strategi**: Kita akan mengelompokkan data per pelanggan (CustomerID), lalu menghitung:
- `total_belanja` — total uang yang dibelanjakan pelanggan
- `jumlah_transaksi` — berapa kali pelanggan bertransaksi
- `jumlah_produk` — berapa banyak produk berbeda yang dibeli
- `rata_rata_belanja` — rata-rata belanja per transaksi

**Label**: Pelanggan dikategorikan **loyal** jika jumlah transaksinya di atas median, selain itu **tidak loyal**.

In [ ]:
# ============================================================
# FEATURE ENGINEERING: Menghitung fitur per pelanggan
# ============================================================
# Kita mengelompokkan data per CustomerID, lalu menghitung metrik-metrik
# yang mencerminkan perilaku belanja pelanggan
# ============================================================

# Kelompokkan data per CustomerID dan hitung metrik perilaku belanja
customer_df = df.groupby('CustomerID').agg(
    total_belanja=('TotalAmount', 'sum'),           # Total uang yang dibelanjakan
    jumlah_transaksi=('InvoiceNo', 'nunique'),      # Jumlah transaksi unik
    jumlah_produk=('StockCode', 'nunique'),         # Jumlah produk berbeda yang dibeli
    rata_rata_qty=('Quantity', 'mean'),             # Rata-rata jumlah item per baris transaksi
    rata_rata_belanja=('TotalAmount', 'mean'),      # Rata-rata belanja per baris transaksi
).reset_index()

print(f"✅ Feature engineering selesai!")
print(f"   Jumlah pelanggan: {len(customer_df):,}")
print(f"\n📋 Statistik fitur:")
customer_df.describe()

In [ ]:
# ============================================================
# MEMBUAT LABEL (TARGET): Loyal vs Tidak Loyal
# ============================================================
# Kriteria: pelanggan dengan jumlah transaksi DI ATAS median = LOYAL
#           pelanggan dengan jumlah transaksi DI BAWAH/SAAT median = TIDAK LOYAL
# Mengapa median? Karena lebih tahan terhadap outlier dibanding mean
# ============================================================

# Hitung median dari jumlah transaksi
median_transaksi = customer_df['jumlah_transaksi'].median()
print(f"📊 Median jumlah transaksi: {median_transaksi}")

# Buat kolom label: 1 = Loyal, 0 = Tidak Loyal
# np.where(kondisi, nilai_jika_benar, nilai_jika_salah)
customer_df['label_loyal'] = np.where(
    customer_df['jumlah_transaksi'] > median_transaksi,  # Kondisi: transaksi > median
    1,                                                    # Jika benar: Loyal (1)
    0                                                     # Jika salah: Tidak Loyal (0)
)

# Tampilkan distribusi label
jumlah_loyal = customer_df['label_loyal'].sum()
jumlah_tidak_loyal = len(customer_df) - jumlah_loyal

print(f"\n📋 Distribusi Label:")
print(f"   Loyal (1)       : {jumlah_loyal} pelanggan ({jumlah_loyal/len(customer_df)*100:.1f}%)")
print(f"   Tidak Loyal (0) : {jumlah_tidak_loyal} pelanggan ({jumlah_tidak_loyal/len(customer_df)*100:.1f}%)")

In [ ]:
# ============================================================
# MEMISAHKAN FITUR (X) DAN LABEL (y)
# ============================================================
# X = fitur/kolom yang digunakan untuk memprediksi (variabel independen)
# y = label/target yang ingin diprediksi (variabel dependen)
# CustomerID TIDAK dimasukkan sebagai fitur karena hanya identifier, bukan prediktor
# jumlah_transaksi TIDAK dimasukkan karena itu dasar pembuatan label (bisa data leakage)
# ============================================================

# Kolom fitur yang digunakan untuk prediksi
fitur = ['total_belanja', 'jumlah_produk', 'rata_rata_qty', 'rata_rata_belanja']

X = customer_df[fitur]                  # Fitur (variabel independen)
y = customer_df['label_loyal']          # Label (variabel dependen: 0 atau 1)

print(f"✅ Fitur dan label berhasil dipisahkan!")
print(f"   Jumlah fitur: {len(fitur)}")
print(f"   Nama fitur: {fitur}")
print(f"   Bentuk X: {X.shape}")
print(f"   Bentuk y: {y.shape}")
print(f"\n📋 Contoh data fitur:")
X.head()

## Step 4: Membagi Data Menjadi Data Latih dan Data Uji

Sebelum melatih model, data harus dibagi menjadi dua bagian:
- **Data Latih (Train)**: digunakan untuk "mengajarkan" model (80% data)
- **Data Uji (Test)**: digunakan untuk menguji seberapa baik model memprediksi data baru (20% data)

Pembagian ini penting agar kita bisa mengukur kemampuan model pada data yang **belum pernah dilihat** sebelumnya.

In [ ]:
# ============================================================
# MEMBAGI DATA: 80% LATIH, 20% UJI
# ============================================================
# train_test_split membagi data secara acak dengan proporsi yang ditentukan
# test_size=0.2 = 20% data untuk pengujian
# random_state=42 = seed acak agar hasil konsisten (reproducible)
# stratify=y = memastikan proporsi label 0 dan 1 sama di data latih dan uji
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,                          # Fitur
    y,                          # Label
    test_size=0.2,              # 20% untuk data uji
    random_state=42,            # Seed untuk reproducibility
    stratify=y                  # Jaga proporsi label seimbang
)

print(f"✅ Data berhasil dibagi!")
print(f"   Data LATIH: {X_train.shape[0]:,} baris ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Data UJI  : {X_test.shape[0]:,} baris ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n📋 Distribusi label di data latih:")
print(f"   Loyal (1)      : {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"   Tidak Loyal (0): {len(y_train)-y_train.sum()} ({(len(y_train)-y_train.sum())/len(y_train)*100:.1f}%)")
print(f"\n📋 Distribusi label di data uji:")
print(f"   Loyal (1)      : {y_test.sum()} ({y_test.sum()/len(y_test)*100:.1f}%)")
print(f"   Tidak Loyal (0): {len(y_test)-y_test.sum()} ({(len(y_test)-y_test.sum())/len(y_test)*100:.1f}%)")

## Step 5: Membangun Model Decision Tree (C4.5)

**Algoritma C4.5** menggunakan **Information Gain** berbasis **Entropy** untuk memilih atribut terbaik sebagai node pemecah di setiap level pohon.

Di scikit-learn, C4.5 setara dengan `DecisionTreeClassifier(criterion='entropy')`.

**Konsep Entropy**:
- Mengukur "ketidakmurnian" (impurity) dari suatu kumpulan data
- Entropy = 0 → data murni (semua satu kelas)
- Entropy = 1 → data paling tidak murni (50:50 dua kelas)
- Rumus: Entropy = -Σ(p_i × log₂(p_i))

In [ ]:
# ============================================================
# MEMBANGUN MODEL DECISION TREE (C4.5)
# ============================================================
# criterion='entropy' = menggunakan Information Gain (setara C4.5)
#   Entropy mengukur ketidakmurnian data:
#   - Entropy rendah = data murni (satu kelas dominan)
#   - Entropy tinggi = data campuran (kelas seimbang)
#   - Rumus: Entropy = -Σ(p_i × log₂(p_i))
#
# max_depth=5 = batas kedalaman pohon maksimal 5 level
#   Tujuan: mencegah overfitting (model terlalu hafal data latih)
#   Semakin dalam pohon, semakin kompleks model → risiko overfitting
#
# random_state=42 = seed acak agar hasil konsisten
# ============================================================

model_dt = DecisionTreeClassifier(
    criterion='entropy',    # Menggunakan entropy (setara algoritma C4.5)
    max_depth=5,            # Batas kedalaman pohon agar tidak overfitting
    random_state=42         # Seed untuk hasil yang konsisten
)

# Latih model menggunakan data latih
# Model akan "belajar" pola dari fitur dan label di data latih
model_dt.fit(X_train, y_train)

print("✅ Model Decision Tree (C4.5) berhasil dilatih!")
print(f"   Kedalaman pohon aktual: {model_dt.get_depth()}")
print(f"   Jumlah daun (leaf): {model_dt.get_n_leaves()}")
print(f"   Jumlah fitur yang digunakan: {model_dt.n_features_in_}")

## Step 6: Prediksi dan Evaluasi Model

Setelah model dilatih, kita menguji kemampuannya pada **data uji** (data yang belum pernah dilihat model).  
Metrik evaluasi yang digunakan:
- **Accuracy**: persentase prediksi yang benar dari seluruh data
- **Precision**: dari semua prediksi "loyal", berapa yang benar-benar loyal
- **Recall**: dari semua pelanggan loyal sesungguhnya, berapa yang berhasil diprediksi
- **F1-Score**: rata-rata harmonis precision dan recall

In [ ]:
# ============================================================
# PREDIKSI PADA DATA UJI
# ============================================================
# model.predict() menghasilkan prediksi label (0 atau 1) untuk setiap baris data uji
# ============================================================

y_pred = model_dt.predict(X_test)

print("✅ Prediksi selesai!")
print(f"   Jumlah data uji: {len(y_test):,}")
print(f"   Prediksi Loyal (1): {y_pred.sum()}")
print(f"   Prediksi Tidak Loyal (0): {len(y_pred) - y_pred.sum()}")

In [ ]:
# ============================================================
# EVALUASI MODEL: ACCURACY, PRECISION, RECALL, F1-SCORE
# ============================================================
# Accuracy = (TP + TN) / (TP + TN + FP + FN) → seberapa benar keseluruhan
# Precision = TP / (TP + FP) → dari prediksi positif, berapa yang benar
# Recall = TP / (TP + FN) → dari data positif aktual, berapa yang tertangkap
# F1-Score = 2 × (Precision × Recall) / (Precision + Recall) → keseimbangan
# ============================================================

# Hitung akurasi
akurasi = accuracy_score(y_test, y_pred)
print(f"🎯 Akurasi Model: {akurasi:.4f} ({akurasi*100:.2f}%)")

# Tampilkan classification report lengkap
print(f"\n📋 Classification Report:")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=['Tidak Loyal (0)', 'Loyal (1)']
))

In [ ]:
# ============================================================
# CONFUSION MATRIX (MATRIKS KEBINGUNGAN)
# ============================================================
# Confusion Matrix menunjukkan perbandingan prediksi vs nilai aktual:
#
#                 Prediksi
#                 0       1
# Aktual 0  [  TN    FP  ]    TN = True Negative  (benar prediksi tidak loyal)
#        1  [  FN    TP  ]    FP = False Positive  (salah prediksi loyal)
#                           FN = False Negative  (salah prediksi tidak loyal)
#                           TP = True Positive  (benar prediksi loyal)
# ============================================================

cm = confusion_matrix(y_test, y_pred)

# Visualisasi confusion matrix dengan heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Loyal', 'Loyal'],
            yticklabels=['Tidak Loyal', 'Loyal'],
            ax=ax)
ax.set_xlabel('Prediksi', fontsize=12, fontweight='bold')
ax.set_ylabel('Aktual', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix — Decision Tree (C4.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Jelaskan isi confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"📋 Penjelasan Confusion Matrix:")
print(f"   True Negative  (TN) = {tn} → Benar diprediksi Tidak Loyal")
print(f"   False Positive (FP) = {fp} → Salah diprediksi Loyal (padahal Tidak Loyal)")
print(f"   False Negative (FN) = {fn} → Salah diprediksi Tidak Loyal (padahal Loyal)")
print(f"   True Positive  (TP) = {tp} → Benar diprediksi Loyal")

## Step 7: Visualisasi Pohon Keputusan (Decision Tree)

Pohon keputusan menunjukkan alur logika model dalam memprediksi.  
Setiap node menampilkan:
- **Kondisi pemecahan** (misal: total_belanja ≤ 3000)
- **Entropy** (tingkat ketidakmurnian)
- **Jumlah sampel** di node tersebut
- **Distribusi kelas** [Tidak Loyal, Loyal]

In [ ]:
# ============================================================
# VISUALISASI POHON KEPUTUSAN (DECISION TREE)
# ============================================================
# Pohon menunjukkan alur logika model:
# - Setiap node internal: kondisi pemecahan (misal: total_belanja ≤ 3000)
# - Setiap daun (leaf): keputusan akhir (Loyal atau Tidak Loyal)
# - Entropy: tingkat ketidakmurnian di node tersebut
# - Samples: jumlah data di node
# - Value: distribusi kelas [Tidak Loyal, Loyal]
# ============================================================

fig, ax = plt.subplots(figsize=(20, 10))

# tree.plot_tree menggambar pohon keputusan
tree.plot_tree(
    model_dt,
    feature_names=fitur,                     # Nama fitur di setiap node
    class_names=['Tidak Loyal', 'Loyal'],     # Nama kelas di daun
    filled=True,                              # Warna node sesuai kelas dominan
    rounded=True,                             # Sudut membulat
    fontsize=10,                              # Ukuran font
    ax=ax
)

ax.set_title('Visualisasi Pohon Keputusan — Algoritma C4.5 (Decision Tree)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Interpretasi: Setiap node memecah data berdasarkan fitur yang paling mengurangi entropy.")
print("   Warna ORANYE = dominan Tidak Loyal, warna BIRU = dominan Loyal.")

In [ ]:
# ============================================================
# FEATURE IMPORTANCE: FITUR MANA YANG PALING PENTING?
# ============================================================
# feature_importances_ menunjukkan seberapa besar kontribusi tiap fitur
# terhadap keputusan model. Semakin tinggi nilainya, semakin penting fitur tersebut.
# Ini salah satu keunggulan Decision Tree: mudah diinterpretasi!
# ============================================================

importance = pd.DataFrame({
    'Fitur': fitur,
    'Importance': model_dt.feature_importances_
}).sort_values('Importance', ascending=False)

print("📋 Feature Importance (dari yang paling penting):")
print("=" * 50)
for idx, row in importance.iterrows():
    print(f"   {row['Fitur']:25s} → {row['Importance']:.4f} ({row['Importance']*100:.1f}%)")

# Visualisasi feature importance
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(importance['Fitur'], importance['Importance'], color='#2ecc71')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance — Decision Tree (C4.5)', fontsize=14, fontweight='bold')
ax.invert_yaxis()  # Fitur terpenting di atas
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2.,
            f'{width:.3f}', va='center', fontsize=11)
plt.tight_layout()
plt.show()

## Step 8: Interpretasi Hasil & Rekomendasi Bisnis

Kita sudah mendapatkan model klasifikasi yang bisa memprediksi apakah pelanggan loyal atau tidak.  
Sekarang kita interpretasikan hasilnya dalam konteks bisnis retail.

In [ ]:
# ============================================================
# INTERPRETASI HASIL & REKOMENDASI BISNIS
# ============================================================
# Kita rangkum temuan dari model klasifikasi dan kaitkan dengan
# konteks bisnis retail (aspek Kemanfaatan/Renstra rubrik)
# ============================================================

print("=" * 70)
print("📊 INTERPRETASI HASIL KLASIFIKASI C4.5")
print("=" * 70)
print()
print(f"1. AKURASI MODEL: {akurasi*100:.2f}%")
print(f"   → Model mampu memprediksi loyalitas pelanggan dengan cukup baik")
print()
print(f"2. FITUR PALING PENTING: {importance.iloc[0]['Fitur']}")
print(f"   → Ini artinya {importance.iloc[0]['Fitur']} adalah faktor utama penentu loyalitas")
print()
print("3. REKOMENDASI BISNIS RETAIL:")
print("   a) Identifikasi pelanggan loyal → berikan program membership/discount khusus")
print("   b) Pelanggan tidak loyal → kirim promosi targeted untuk meningkatkan engagement")
print("   c) Fokus pada fitur terpenting → jika total_belanja paling penting,")
print("      buat program reward berdasarkan nilai belanja")
print("   d) Gunakan model ini secara berkala untuk memantau perubahan loyalitas pelanggan")
print()
print("4. KELEBIHAN C4.5 (DECISION TREE):")
print("   - Mudah diinterpretasi (bisa divisualisasikan sebagai pohon)")
print("   - Tidak memerlukan normalisasi/scaling data")
print("   - Bisa menangani data kategorik dan numerik")
print()
print("5. KEKURANGAN C4.5 (DECISION TREE):")
print("   - Rentan terhadap overfitting jika pohon terlalu dalam")
print("   - Sensitif terhadap perubahan kecil pada data")
print("   - Perlu pruning atau batasan kedalaman untuk generalisasi yang baik")